# 特徴量エンジニアリング実験

- **ベース**: `train_baseline.ipynb` と同じパイプライン（34＋3C3＋テキストメタ ＝ 38 特徴）。ここで**追加特徴量**を試し、時系列 CV で比較する。
- **検証**: 時系列CV（2013〜2016 を val）。追加列は「実験用」セルで定義し、`FEATURES` に足してから CV を実行。
- **方針**: `docs/02_FEATURE_ENGINEERING.md` を参照（置き換えより「足す」、掛け合わせは少数ずつ試す）。

In [1]:
import os
import random
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
import lightgbm as lgb
import warnings
warnings.filterwarnings("ignore")

from preprocess import load_train_test
from feature_engineering import create_features

print("Setup complete.")

Setup complete.


In [2]:
def seed_everything(seed=42):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)

seed_everything(42)

In [3]:
train, test = load_train_test()
print(f"Train: {len(train):,}, Test: {len(test):,}")

Train: 653,507, Test: 40,716


In [4]:
train = create_features(train)
test = create_features(test)

# モデルにそのまま使う列のうち、object は category に（欠損は "missing"）
for col in ["rotten_tomatoes_link", "critic_name", "movie_title", "movie_info", "directors", "authors", "actors", "production_company"]:
    if col in train.columns and col in test.columns:
        train[col] = train[col].fillna("missing").astype("category")
        test[col] = test[col].fillna("missing").astype("category")
# 数値で欠損がある列は train の中央値で埋める
for col in ["runtime"]:
    if col in train.columns and col in test.columns and train[col].isna().any():
        med = train[col].median()
        train[col] = train[col].fillna(med)
        test[col] = test[col].fillna(med)

print("特徴量作成完了.")

特徴量作成完了.


In [5]:
# 3C3 パターン: 時系列TE（critic_name, production_company）＋ TE 3ビン ＋ runtime_bin, movie_age_bin, release_decade
def _ts_te_col(df_tr, df_te, col, target_name="target", m=10):
    """時系列TE: tr は「その行より前」のみで平均、te は tr のカテゴリ別スムージング平均でマッピング。"""
    global_mean = float(df_tr[target_name].mean())
    tr_s = df_tr.sort_values("review_date")
    g = tr_s.groupby(col)[target_name]
    past_sum = g.cumsum() - tr_s[target_name]
    past_count = g.cumcount()
    te_tr = np.where(past_count > 0, (past_sum + m * global_mean) / (past_count + m), global_mean)
    ser_tr = pd.Series(te_tr, index=tr_s.index)
    agg = df_tr.groupby(col)[target_name].agg(["mean", "count"])
    agg["smooth"] = (agg["count"] * agg["mean"] + m * global_mean) / (agg["count"] + m)
    map_ = agg["smooth"].to_dict()
    te_arr = df_te[col].astype(str).map(map_).fillna(global_mean).values if df_te is not None and len(df_te) else np.array([])
    return ser_tr, te_arr

for col in ["critic_name", "production_company"]:
    if col not in train.columns:
        continue
    st, ta = _ts_te_col(train, test, col, "target", m=10)
    train[f"{col}_te_ts"] = st.reindex(train.index).fillna(train["target"].mean())
    test[f"{col}_te_ts"] = ta

for c in ["critic_name_te_ts", "production_company_te_ts"]:
    if c not in train.columns:
        continue
    train[c + "_bin"] = pd.cut(train[c], bins=[0, 0.33, 0.67, 1.01], labels=[0, 1, 2]).astype(float).fillna(1)
    test[c + "_bin"] = pd.cut(test[c], bins=[0, 0.33, 0.67, 1.01], labels=[0, 1, 2]).astype(float).fillna(1)

train["runtime_bin"] = pd.cut(train["runtime"], bins=[0, 90, 120, 150, 1000], labels=[0, 1, 2, 3]).astype(float).fillna(1)
test["runtime_bin"] = pd.cut(test["runtime"], bins=[0, 90, 120, 150, 1000], labels=[0, 1, 2, 3]).astype(float).fillna(1)
def _movie_age_bin(x):
    if pd.isna(x) or x < 0: return 0
    if x < 365: return 1
    if x < 365 * 5: return 2
    if x < 365 * 20: return 3
    return 4
train["movie_age_bin"] = train["movie_age_days"].apply(_movie_age_bin)
test["movie_age_bin"] = test["movie_age_days"].apply(_movie_age_bin)
train["release_decade"] = (train["release_year"] // 10 * 10).fillna(1990)
test["release_decade"] = (test["release_year"] // 10 * 10).fillna(1990)
print("3C3 特徴量追加完了.")

3C3 特徴量追加完了.


In [7]:
# テキストメタ追加（movie_info_and_title_meta: train_text_experiments で最良だった設定）
from experiment_encodings import movie_info_meta
movie_info_meta(train, pd.DataFrame(), test)
train["movie_title_len"] = train["movie_title"].astype(str).str.len()
test["movie_title_len"] = test["movie_title"].astype(str).str.len()
train["movie_title_word_count"] = train["movie_title"].astype(str).str.split().str.len().fillna(0).astype(int)
test["movie_title_word_count"] = test["movie_title"].astype(str).str.split().str.len().fillna(0).astype(int)
print("テキストメタ（movie_info_and_title_meta）追加完了.")

テキストメタ（movie_info_and_title_meta）追加完了.


In [8]:
# ベース特徴量（3C3 ＋ テキストメタ）。実験用追加列は次のセルで EXTRA_FEATURES に足す。
BASE_FEATURES = [
    "rotten_tomatoes_link",
    "critic_name",
    "top_critic",
    "publisher_name",
    #映画情報
    "movie_title",
    "movie_info",
    "content_rating",
    "directors",
    "authors",
    "actors",
    "runtime",
    "production_company",
    #レビュー系
    "review_year",
    "review_month",
    "review_dayofweek",
    #リリース系
    "release_year",
    "release_month",
    "release_dayofweek",
    "movie_age_days",
    #カテゴリ特徴量
    "genre_Drama",
    "genre_Comedy",
    "genre_Action",
    "genre_Mystery",
    "genre_Fantasy",
    "genre_Romance",
    "genre_Horror",
    "genre_Documentary",
    # 3C3 追加
    "critic_name_te_ts",
    "production_company_te_ts",
    "critic_name_te_ts_bin",
    "production_company_te_ts_bin",
    "runtime_bin",
    "movie_age_bin",
    "release_decade",
    # テキストメタ（movie_info_and_title_meta）
    "movie_info_len",
    "movie_info_word_count",
    "movie_title_len",
    "movie_title_word_count",
]
# 追加特徴量を試すときは次の「実験: 追加特徴量」セルで EXTRA_FEATURES を設定してから CV を実行
FEATURES = BASE_FEATURES.copy()
print(f"使用特徴量: {len(FEATURES)}個 (ベース: {len(BASE_FEATURES)}個)")

使用特徴量: 38個 (ベース: 38個)


In [12]:
train

,ID,rotten_tomatoes_link,critic_name,top_critic,publisher_name,review_date,movie_title,movie_info,content_rating,genres,...,genre_Romance,genre_Horror,genre_Documentary,critic_name_te_ts,production_company_te_ts,critic_name_te_ts_bin,production_company_te_ts_bin,runtime_bin,movie_age_bin,release_decade
0,0,m/0814255,Andrew L. Urban,False,Urban Cinefile,2010-02-06,Percy Jackson & the Olympians: The Lightning T...,"Always trouble-prone, the life of teenager Per...",PG,"Action & Adventure, Comedy, Drama, Science Fic...",...,0,0,0,0.803451,0.489462,2.0,1.0,1.0,0,2010.0
1,1,m/0814255,Louise Keller,False,Urban Cinefile,2010-02-06,Percy Jackson & the Olympians: The Lightning T...,"Always trouble-prone, the life of teenager Per...",PG,"Action & Adventure, Comedy, Drama, Science Fic...",...,0,0,0,0.820255,0.489491,2.0,1.0,1.0,0,2010.0
2,2,m/0814255,David Germain,True,Associated Press,2010-02-10,Percy Jackson & the Olympians: The Lightning T...,"Always trouble-prone, the life of teenager Per...",PG,"Action & Adventure, Comedy, Drama, Science Fic...",...,0,0,0,0.413455,0.489582,1.0,1.0,1.0,0,2010.0
3,3,m/0814255,Nick Schager,False,Slant Magazine,2010-02-10,Percy Jackson & the Olympians: The Lightning T...,"Always trouble-prone, the life of teenager Per...",PG,"Action & Adventure, Comedy, Drama, Science Fic...",...,0,0,0,0.360928,0.489637,1.0,1.0,1.0,0,2010.0
4,4,m/0814255,Bill Goodykoontz,True,Arizona Republic,2010-02-10,Percy Jackson & the Olympians: The Lightning T...,"Always trouble-prone, the life of teenager Per...",PG,"Action & Adventure, Comedy, Drama, Science Fic...",...,0,0,0,0.682583,0.489549,2.0,1.0,1.0,0,2010.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
653502,653502,m/zulu_dawn,Brandon Judell,False,PopcornQ,2005-08-14,Zulu Dawn,Sir Henry Bartle Frere's (John Mills) vastly o...,PG,"Action & Adventure, Art House & International,...",...,0,0,0,0.702341,0.562785,2.0,1.0,2.0,4,1970.0
653503,653503,m/zulu_dawn,Cole Smithey,False,ColeSmithey.com,2005-11-01,Zulu Dawn,Sir Henry Bartle Frere's (John Mills) vastly o...,PG,"Action & Adventure, Art House & International,...",...,0,0,0,0.820266,0.599219,2.0,1.0,2.0,4,1970.0
653504,653504,m/zulu_dawn,Ken Hanke,False,"Mountain Xpress (Asheville, NC)",2007-03-07,Zulu Dawn,Sir Henry Bartle Frere's (John Mills) vastly o...,PG,"Action & Adventure, Art House & International,...",...,0,0,0,0.711286,0.630048,2.0,1.0,2.0,4,1970.0
653505,653505,m/zulu_dawn,Dennis Schwartz,False,Dennis Schwartz Movie Reviews,2010-09-16,Zulu Dawn,Sir Henry Bartle Frere's (John Mills) vastly o...,PG,"Action & Adventure, Art House & International,...",...,0,0,0,0.602209,0.656474,1.0,1.0,2.0,4,1970.0


## 15パターン追加特徴量の作成とCV比較

以下で **15 種類の追加特徴量**を train/test に作成し、それぞれ「ベース + その 1 パターン」で時系列 CV を回して AUC を比較する。  
方針: 批評家TE・制作会社TE × ジャンル/年代/ビン、およびメタ同士の交互作用（`docs/02_FEATURE_ENGINEERING.md`）。

In [10]:
# ========== 15パターン追加特徴量の作成（train / test 両方） ==========
# P01: 批評家TE × ジャンル（Drama）
train["critic_te_x_genre_Drama"] = train["critic_name_te_ts"] * train["genre_Drama"]
test["critic_te_x_genre_Drama"] = test["critic_name_te_ts"] * test["genre_Drama"]

# P02: 批評家TE × ジャンル（Comedy）
train["critic_te_x_genre_Comedy"] = train["critic_name_te_ts"] * train["genre_Comedy"]
test["critic_te_x_genre_Comedy"] = test["critic_name_te_ts"] * test["genre_Comedy"]

# P03: 批評家TE × ジャンル（Documentary）
train["critic_te_x_genre_Documentary"] = train["critic_name_te_ts"] * train["genre_Documentary"]
test["critic_te_x_genre_Documentary"] = test["critic_name_te_ts"] * test["genre_Documentary"]

# P04: 制作会社TE × 年代（正規化: 1950→0, 2020→0.7）
_release_norm = (train["release_decade"].min(), train["release_decade"].max())
train["release_decade_norm"] = (train["release_decade"] - _release_norm[0]) / max(_release_norm[1] - _release_norm[0], 1)
test["release_decade_norm"] = (test["release_decade"] - _release_norm[0]) / max(_release_norm[1] - _release_norm[0], 1)
train["prod_te_x_release_decade"] = train["production_company_te_ts"] * train["release_decade_norm"]
test["prod_te_x_release_decade"] = test["production_company_te_ts"] * test["release_decade_norm"]

# P05: 批評家TE × runtime_bin
train["critic_te_x_runtime_bin"] = train["critic_name_te_ts"] * train["runtime_bin"]
test["critic_te_x_runtime_bin"] = test["critic_name_te_ts"] * test["runtime_bin"]

# P06: 制作会社TE × runtime_bin
train["prod_te_x_runtime_bin"] = train["production_company_te_ts"] * train["runtime_bin"]
test["prod_te_x_runtime_bin"] = test["production_company_te_ts"] * test["runtime_bin"]

# P07: 批評家TE × movie_age_bin
train["critic_te_x_movie_age_bin"] = train["critic_name_te_ts"] * train["movie_age_bin"]
test["critic_te_x_movie_age_bin"] = test["critic_name_te_ts"] * test["movie_age_bin"]

# P08: 制作会社TE × ジャンル（Drama）
train["prod_te_x_genre_Drama"] = train["production_company_te_ts"] * train["genre_Drama"]
test["prod_te_x_genre_Drama"] = test["production_company_te_ts"] * test["genre_Drama"]

# P09: release_decade_norm × Documentary（年代とドキュメンタリーの相性）
train["release_decade_x_genre_Documentary"] = train["release_decade_norm"] * train["genre_Documentary"]
test["release_decade_x_genre_Documentary"] = test["release_decade_norm"] * test["genre_Documentary"]

# P10: top_critic × 批評家TE（トップ批評家かどうかと傾向の交互作用）
train["top_critic_x_critic_te"] = train["top_critic"].astype(int) * train["critic_name_te_ts"]
test["top_critic_x_critic_te"] = test["top_critic"].astype(int) * test["critic_name_te_ts"]

# P11: runtime_bin × ジャンル（Drama）（長尺とドラマの相性）
train["runtime_bin_x_genre_Drama"] = train["runtime_bin"] * train["genre_Drama"]
test["runtime_bin_x_genre_Drama"] = test["runtime_bin"] * test["genre_Drama"]

# P12: movie_age_bin × Documentary（古い映画とドキュメンタリー）
train["movie_age_bin_x_genre_Documentary"] = train["movie_age_bin"] * train["genre_Documentary"]
test["movie_age_bin_x_genre_Documentary"] = test["movie_age_bin"] * test["genre_Documentary"]

# P13: 批評家TE × タイトル長（正規化）
_title_len_max = train["movie_title_len"].max() or 1
train["movie_title_len_norm"] = train["movie_title_len"] / _title_len_max
test["movie_title_len_norm"] = test["movie_title_len"].clip(upper=_title_len_max) / _title_len_max
train["critic_te_x_title_len"] = train["critic_name_te_ts"] * train["movie_title_len_norm"]
test["critic_te_x_title_len"] = test["critic_name_te_ts"] * test["movie_title_len_norm"]

# P14: 制作会社TE × movie_age_bin
train["prod_te_x_movie_age_bin"] = train["production_company_te_ts"] * train["movie_age_bin"]
test["prod_te_x_movie_age_bin"] = test["production_company_te_ts"] * test["movie_age_bin"]

# P15: レビュー年（正規化）× 批評家TE（時間トレンド）
_ry_min, _ry_max = train["review_year"].min(), train["review_year"].max()
train["review_year_norm"] = (train["review_year"] - _ry_min) / max(_ry_max - _ry_min, 1)
test["review_year_norm"] = (test["review_year"] - _ry_min) / max(_ry_max - _ry_min, 1)
train["review_year_norm_x_critic_te"] = train["review_year_norm"] * train["critic_name_te_ts"]
test["review_year_norm_x_critic_te"] = test["review_year_norm"] * test["critic_name_te_ts"]

print("15パターン分の追加特徴量を作成しました。")

# ========== 実験設定: ベース + 各パターン1つ ==========
FEATURE_EXPERIMENT_CONFIGS = [
    ("base", []),
    ("p01_critic_te_x_genre_Drama", ["critic_te_x_genre_Drama"]),
    ("p02_critic_te_x_genre_Comedy", ["critic_te_x_genre_Comedy"]),
    ("p03_critic_te_x_genre_Documentary", ["critic_te_x_genre_Documentary"]),
    ("p04_prod_te_x_release_decade", ["prod_te_x_release_decade"]),
    ("p05_critic_te_x_runtime_bin", ["critic_te_x_runtime_bin"]),
    ("p06_prod_te_x_runtime_bin", ["prod_te_x_runtime_bin"]),
    ("p07_critic_te_x_movie_age_bin", ["critic_te_x_movie_age_bin"]),
    ("p08_prod_te_x_genre_Drama", ["prod_te_x_genre_Drama"]),
    ("p09_release_decade_x_genre_Doc", ["release_decade_x_genre_Documentary"]),
    ("p10_top_critic_x_critic_te", ["top_critic_x_critic_te"]),
    ("p11_runtime_bin_x_genre_Drama", ["runtime_bin_x_genre_Drama"]),
    ("p12_movie_age_bin_x_genre_Doc", ["movie_age_bin_x_genre_Documentary"]),
    ("p13_critic_te_x_title_len", ["critic_te_x_title_len"]),
    ("p14_prod_te_x_movie_age_bin", ["prod_te_x_movie_age_bin"]),
    ("p15_review_year_x_critic_te", ["review_year_norm_x_critic_te"]),
]

# ========== 時系列CV分割（このセル内で完結） ==========
VAL_YEARS = [2013, 2014, 2015, 2016]
time_splits = []
for vy in VAL_YEARS:
    tr_idx = np.where(train["review_year"] < vy)[0]
    val_idx = np.where(train["review_year"] == vy)[0]
    if len(val_idx) > 0:
        time_splits.append((tr_idx, val_idx))

lgb_params = {
    "objective": "binary", "metric": "auc", "boosting_type": "gbdt",
    "n_estimators": 100, "learning_rate": 0.1, "num_leaves": 31,
    "random_state": 42, "verbosity": -1,
}
y = train["target"].values

# ========== 各設定でCV実行 ==========
results_fe = []
for config_name, extra_cols in FEATURE_EXPERIMENT_CONFIGS:
    feats = BASE_FEATURES + extra_cols
    X = train[feats]
    X_te = test[feats]
    fold_scores = []
    for tr_idx, val_idx in time_splits:
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y[tr_idx], y[val_idx]
        model = lgb.LGBMClassifier(**lgb_params)
        model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], callbacks=[lgb.early_stopping(20, verbose=False)])
        auc = roc_auc_score(y_val, model.predict_proba(X_val)[:, 1])
        fold_scores.append(auc)
    mean_auc = np.mean(fold_scores)
    results_fe.append({"config": config_name, "n_feat": len(feats), "CV_AUC": mean_auc, "std": np.std(fold_scores)})
    print(f"  {config_name}: AUC={mean_auc:.4f} ± {np.std(fold_scores):.4f}")

res_df = pd.DataFrame(results_fe)
base_auc = res_df.loc[res_df["config"] == "base", "CV_AUC"].iloc[0]
res_df["diff_vs_base"] = res_df["CV_AUC"] - base_auc
res_df = res_df.sort_values("CV_AUC", ascending=False).reset_index(drop=True)

# 全パターンの精度差をテキストで表示（必ず出力に残る）
print("\n========== 全16パターン CV結果（ベースとの差分付き） ==========")
print(res_df[["config", "n_feat", "CV_AUC", "std", "diff_vs_base"]].to_string(index=True))
print(f"\nベース AUC = {base_auc:.4f}")

display(res_df)
best_row = res_df[res_df["config"] != "base"].iloc[0] if (res_df["config"] != "base").any() else None
if best_row is not None and best_row["CV_AUC"] > res_df[res_df["config"] == "base"].iloc[0]["CV_AUC"]:
    best_extra = [c for _, c in FEATURE_EXPERIMENT_CONFIGS if _ == best_row["config"]][0]
    FEATURES = BASE_FEATURES + best_extra
    print(f"\nベスト: {best_row['config']} (CV_AUC={best_row['CV_AUC']:.4f}). FEATURES を更新しました。以下の提出セルで利用できます。")
else:
    print("\nベースが最良のため FEATURES は変更しません。")

# 提出セル用（全 train で学習するときに使う）
X = train[FEATURES]
X_test = test[FEATURES]

15パターン分の追加特徴量を作成しました。
  base: AUC=0.7602 ± 0.0056
  p01_critic_te_x_genre_Drama: AUC=0.7604 ± 0.0054
  p02_critic_te_x_genre_Comedy: AUC=0.7602 ± 0.0056
  p03_critic_te_x_genre_Documentary: AUC=0.7612 ± 0.0049
  p04_prod_te_x_release_decade: AUC=0.7603 ± 0.0056
  p05_critic_te_x_runtime_bin: AUC=0.7604 ± 0.0042
  p06_prod_te_x_runtime_bin: AUC=0.7594 ± 0.0054
  p07_critic_te_x_movie_age_bin: AUC=0.7606 ± 0.0042
  p08_prod_te_x_genre_Drama: AUC=0.7600 ± 0.0051
  p09_release_decade_x_genre_Doc: AUC=0.7603 ± 0.0052
  p10_top_critic_x_critic_te: AUC=0.7602 ± 0.0056
  p11_runtime_bin_x_genre_Drama: AUC=0.7605 ± 0.0059
  p12_movie_age_bin_x_genre_Doc: AUC=0.7602 ± 0.0056
  p13_critic_te_x_title_len: AUC=0.7602 ± 0.0056
  p14_prod_te_x_movie_age_bin: AUC=0.7604 ± 0.0057
  p15_review_year_x_critic_te: AUC=0.7608 ± 0.0050

========== 全16パターン CV結果（ベースとの差分付き） ==========
                               config  n_feat    CV_AUC       std  diff_vs_base
0   p03_critic_te_x_genre_Documentary      39  0

,config,n_feat,CV_AUC,std,diff_vs_base
0,p03_critic_te_x_genre_Documentary,39,0.761211,0.004887,0.000991
1,p15_review_year_x_critic_te,39,0.760839,0.005026,0.000619
2,p07_critic_te_x_movie_age_bin,39,0.760624,0.004206,0.000404
3,p11_runtime_bin_x_genre_Drama,39,0.760460,0.005867,0.000240
4,p05_critic_te_x_runtime_bin,39,0.760438,0.004171,0.000217
5,p14_prod_te_x_movie_age_bin,39,0.760429,0.005661,0.000208
6,p01_critic_te_x_genre_Drama,39,0.760409,0.005430,0.000188
7,p04_prod_te_x_release_decade,39,0.760334,0.005648,0.000113
8,p09_release_decade_x_genre_Doc,39,0.760328,0.005158,0.000107
9,base,38,0.760220,0.005580,0.000000



ベスト: p03_critic_te_x_genre_Documentary (CV_AUC=0.7612). FEATURES を更新しました。以下の提出セルで利用できます。


## 段階的組み合わせ実験（改善した9パターンを順に足す）

単体でベースより良かった 9 パターンを、1個→2個→…→9個と段階的に足し、CV AUC がどこまで伸びるかを比較する。**上のセル実行後に**このセルを実行する。

In [ ]:
# 単体でベースより良かった9パターン（AUC順）の追加列
IMPROVING_COLS = [
    "critic_te_x_genre_Documentary",   # p03
    "review_year_norm_x_critic_te",    # p15
    "critic_te_x_movie_age_bin",        # p07
    "runtime_bin_x_genre_Drama",        # p11
    "critic_te_x_runtime_bin",          # p05
    "prod_te_x_movie_age_bin",          # p14
    "critic_te_x_genre_Drama",          # p01
    "prod_te_x_release_decade",         # p04
    "release_decade_x_genre_Documentary", # p09
]

# ステージ: base → +1個 → +2個 → … → +9個
STAGE_CONFIGS = [("stage0_base", [])]
for k in range(1, len(IMPROVING_COLS) + 1):
    STAGE_CONFIGS.append((f"stage{k}_+{k}patterns", IMPROVING_COLS[:k]))

# 時系列CVは上のセルで定義済み（time_splits, lgb_params, y）
results_stage = []
for stage_name, extra_cols in STAGE_CONFIGS:
    feats = BASE_FEATURES + extra_cols
    X = train[feats]
    fold_scores = []
    for tr_idx, val_idx in time_splits:
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y[tr_idx], y[val_idx]
        model = lgb.LGBMClassifier(**lgb_params)
        model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], callbacks=[lgb.early_stopping(20, verbose=False)])
        auc = roc_auc_score(y_val, model.predict_proba(X_val)[:, 1])
        fold_scores.append(auc)
    mean_auc = np.mean(fold_scores)
    results_stage.append({"stage": stage_name, "n_feat": len(feats), "CV_AUC": mean_auc, "std": np.std(fold_scores)})
    print(f"  {stage_name}: n_feat={len(feats)}, AUC={mean_auc:.4f} ± {np.std(fold_scores):.4f}")

stage_df = pd.DataFrame(results_stage)
stage_df["diff_vs_base"] = stage_df["CV_AUC"] - stage_df.loc[0, "CV_AUC"]
print("\n========== 段階的組み合わせ CV結果 ==========")
print(stage_df.to_string(index=True))
display(stage_df)

  stage0_base: n_feat=38, AUC=0.7602 ± 0.0056
  stage1_+1patterns: n_feat=39, AUC=0.7612 ± 0.0049


In [16]:
# 提出用: 全 train で 1 本学習して test を予測（ベースラインと同じく「全データで学習→予測」）
model_full = lgb.LGBMClassifier(**lgb_params)
model_full.fit(X, y)
final_pred = model_full.predict_proba(X_test)[:, 1]

submission = pd.DataFrame({"ID": test["ID"], "target": final_pred})
submission.to_csv("submission.csv", index=False)
print(f"Saved submission.csv (rows: {len(submission):,}) [全trainで学習した1本のモデルで予測]")

# 特徴量重要度（model_full の gain）
importance_df = pd.DataFrame({
    "feature": FEATURES,
    "importance": model_full.feature_importances_,
}).sort_values("importance", ascending=False)
display(importance_df)
print(f"\n重要度 Top5: {list(importance_df.head(5)['feature'].values)}")

Saved submission.csv (rows: 40,716) [全trainで学習した1本のモデルで予測]


,feature,importance
7,directors,727
8,authors,601
0,rotten_tomatoes_link,488
4,movie_title,411
1,critic_name,211
3,publisher_name,155
9,actors,93
11,production_company,83
27,critic_name_te_ts,54
5,movie_info,41



重要度 Top5: ['directors', 'authors', 'rotten_tomatoes_link', 'movie_title', 'critic_name']
